# Ask_Spectrum support agent — reproduce in ColabRuns the same code as the repo. Colab is just the machine; all logic lives in`support_agent/` and `evaluation/` so it can be explained and modified live.**Order:** setup → data → golden pool → label → eval → judge agreement → report numbers.Runtime: CPU is fine. Everything is LLM-API bound, not GPU bound.

## 1. Setup

In [ ]:
REPO = "https://github.com/punith24246/hiver-support-agent.git"   # <-- your repoimport os, sys, subprocessif not os.path.exists("hiver-support-agent"):    subprocess.run(["git","clone",REPO], check=True)os.chdir("/content/hiver-support-agent")!pip install -q -r requirements.txtsys.path.insert(0, os.getcwd())print(os.getcwd())

In [ ]:
# Keys. Use Colab secrets (key icon, left sidebar) so nothing lands in the notebook.from google.colab import userdataos.environ["GROQ_API_KEY"]   = userdata.get("GROQ_API_KEY")os.environ["KAGGLE_USERNAME"]= userdata.get("KAGGLE_USERNAME")os.environ["KAGGLE_KEY"]     = userdata.get("KAGGLE_KEY")# Sanity: 11 tests, synthetic data, no network, no key used.!AGENT_MOCK=1 python -m pytest -q

## 2. Data~500 MB download, the only slow step. Mount Drive first if you want to cache it across sessions.

In [ ]:
# Optional: cache twcs.csv in Drive so you download it once.USE_DRIVE = Falseif USE_DRIVE:    from google.colab import drive; drive.mount("/content/drive")    os.makedirs("data", exist_ok=True)    src = "/content/drive/MyDrive/twcs.csv"    if os.path.exists(src) and not os.path.exists("data/twcs.csv"):        !cp "{src}" data/twcs.csv!python -m scripts.get_data

In [ ]:
from support_agent.data import load_brand_threads   # adjust if your fn name differsimport pandas as pddf = pd.read_csv("data/twcs.csv", nrows=200_000)print(df[df.author_id=="Ask_Spectrum"].shape, "Spectrum tweets in first 200k rows")df.head(3)

## 3. Build the golden poolStratified by weak intent x time bucket. `--prelabel` has the *judge* model proposea label; you correct it. A blind slice (`--blind`) is held out with no pre-label so youcan measure how much the pre-label anchored you — that is evidence, not decoration.

In [ ]:
!python -m scripts.build_golden_pool --config configs/spectrum.yaml --n 220 --blind 50 --prelabel!head -c 800 data/golden/pool.jsonl

## 4. LabelTwo options. The widget is faster in Colab; the CLI is the same data either way.Budget ~35-45 min for 220 items at ~10 s each. Do it in two sittings and note the break —labeller fatigue is a real caveat for the misleading-number section.

In [ ]:
# Option A: inline labelling widgetimport json, ipywidgets as Wfrom IPython.display import display, clear_outputfrom support_agent.intents import INTENTSPOOL, OUT = "data/golden/pool.jsonl", "data/golden/golden.jsonl"load = lambda p: [json.loads(l) for l in open(p)] if os.path.exists(p) else []pool, done = load(POOL), {r["customer_tweet_id"] for r in load(OUT)}todo = [r for r in pool if r["customer_tweet_id"] not in done]names = [i.name for i in INTENTS]state = {"i": 0}out = W.Output()intent = W.ToggleButtons(options=names, value=names[0])action = W.ToggleButtons(options=["auto", "escalate"], value="auto")note   = W.Text(placeholder="why escalate / edge case note")def show():    with out:        clear_output()        if state["i"] >= len(todo):            print("done"); return        r = todo[state["i"]]        print(f"[{state['i']+1}/{len(todo)}] arm={r.get('sample_arm')} blind={r.get('blind', False)}")        print("-"*70); print(r["customer_text"]); print("-"*70)        if not r.get("blind"):            print("pre-label:", r.get("prelabel_intent"), "|", r.get("prelabel_action"))def save(_):    r = todo[state["i"]]    rec = dict(r, gold_intent=intent.value, gold_action=action.value, note=note.value)    with open(OUT, "a") as f: f.write(json.dumps(rec)+"\n")    note.value = ""; state["i"] += 1; show()btn = W.Button(description="save + next", button_style="success")btn.on_click(save)display(W.VBox([out, intent, action, note, btn])); show()

In [ ]:
# Option B: terminal-style labelling (identical output file)# !python -m scripts.label_cli

## 5. Run the evaluationThis is the deliverable. Agent vs trivial + simple baselines, on the golden set.

In [ ]:
# Smoke it on 25 examples first so a prompt bug doesn't cost 220 calls.!python -m evaluation.run_eval --config configs/spectrum.yaml --limit 25 --no-judge

In [ ]:
# Full run. LLM calls are cached in runs/llm_cache.sqlite, so re-runs are ~40 s.!python -m evaluation.run_eval --config configs/spectrum.yaml

## 6. Judge agreementThe harness is only credible if the judge tracks a human. Score ~60 replies yourself,then report quadratic-weighted kappa and Spearman against the judge. Report it even ifit is mediocre — a stated weak kappa beats an unstated one.

In [ ]:
!python -m scripts.score_replies --run runs/latest --n 60!python -m evaluation.run_eval --config configs/spectrum.yaml   # re-run picks up agreement stats

## 7. Headline numbers

In [ ]:
from IPython.display import Markdownprint(sorted(os.listdir("runs")))display(Markdown(open("runs/latest/summary.md").read()))

## 8. Commit results backThe golden set and human scores must be in the repo — otherwise nobody can reproduceagainst the labels you actually used.

In [ ]:
!git add data/golden/golden.jsonl data/golden/human_scores.jsonl runs/latest docs/!git -c user.email=you@example.com -c user.name=punith24246 commit -m "golden set + eval run"# !git push